# C12-classical-models — Session 2: Linear SVM, Margins, and Hinge Loss

*One 90-minute session. Labels are encoded as $t_i\in\{-1,+1\}$ throughout this session.*


In [ ]:
import numpy as np

SEED = 20260804
ATOL = 1e-10
RTOL = 1e-8
rng = np.random.default_rng(SEED)


## 1. Scores, label encoding, and functional margin

A linear classifier has score $f(x)=w^Tx+b$. Predict $+1$ when $f(x)\ge0$ and $-1$ otherwise;
ties at exactly zero go to $+1$ in this unit. The **functional margin** of labeled row
$(x_i,t_i)$ is $m_i=t_i f(x_i)$. It is positive exactly when the row is correctly classified,
zero on the boundary, and negative when misclassified.

Multiplying both $w$ and $b$ by a positive constant preserves predictions but scales functional
margins. Functional margin therefore mixes boundary geometry with parameter scale.

**Checkpoint 1A.** For $t=-1$ and score $-2.5$, compute $m$ and classify correctness.

**Checkpoint 1B.** What prediction does this unit assign at score zero?


## 2. Geometric margin, hard constraints, and support vectors

The signed perpendicular distance is $t_i(w^Tx_i+b)/\|w\|_2$. Rescale a separable solution so
its closest functional margin is 1. The hard-margin primal is

$$\min_{w,b}\frac12\|w\|_2^2\quad\text{s.t. }t_i(w^Tx_i+b)\ge1.$$

The two canonical margin planes are $f(x)=\pm1$, separated by width $2/\|w\|_2$. Rows with a
**nonzero dual coefficient** are support vectors and pin the maximum-margin boundary. In the usual
nondegenerate hard-margin picture they attain $m_i=1$, while rows with $m_i>1$ lie outside the
margin. But margin equality alone does not certify support-vector status: a degenerate optimum can
have $m_i=1$ and dual coefficient zero. Session 3 derives this caveat from complementary slackness.

**Checkpoint 2A.** If $\|w\|=4$, what is the full margin-band width?

**Checkpoint 2B.** What dual-state criterion defines a support vector, and why is $m_i=1$ alone
insufficient?


## 3. Soft margin, slack, hinge loss, and `C`

Nonseparable data introduce slack $\xi_i\ge0$ with $t_i f(x_i)\ge1-\xi_i$. Eliminating the
smallest valid slack gives hinge loss $h_i=\max(0,1-m_i)$. One common mean-hinge objective is

$$J(w,b)=\frac12\|w\|_2^2+C\frac1N\sum_i\max(0,1-t_i(w^Tx_i+b)).$$

Large `C` prices margin violations heavily and usually fits training labels more aggressively;
small `C` accepts more violations for stronger regularization. In scikit-learn's `SVC`, `C`
uses a sum-loss convention internally, but its qualitative role is the same. Always state the
exact convention before comparing numeric objectives.

Rows with $m_i<1$ have active hinge loss, rows with $m_i>1$ do not, and at $m_i=1$ the hinge is
nondifferentiable. This unit chooses subgradient contribution zero at the kink.

**Checkpoint 3A.** Give hinge losses at margins $-1,0.4,1,2$.

**Checkpoint 3B.** Which direction does training pressure usually move as `C` increases?


## 4. Vectorized objective and one valid subgradient step

Let `active = t * (X @ w + b) < 1`. Away from the kink, a valid subgradient is

$$g_w=w-\frac C N X_A^Tt_A,\qquad g_b=-\frac C N\sum_{i\in A}t_i.$$

The regularizer applies to $w$, not the intercept $b$. A step uses
$w\leftarrow w-\eta g_w$ and $b\leftarrow b-\eta g_b$. The active mask must use signed margin,
not raw score.

**Checkpoint 4A.** Why is `scores < 1` an invalid activity test when negative labels exist?

**Checkpoint 4B.** State the shape of `active` and `grad_w` for `X.shape == (7, 3)`.


In [ ]:
def hinge_objective_and_subgradient(X, t, w, b, C):
    X = np.asarray(X, dtype=np.float64)
    t = np.asarray(t, dtype=np.float64)
    w = np.asarray(w, dtype=np.float64)
    margins = t * (X @ w + b)
    active = margins < 1.0
    objective = 0.5 * (w @ w) + C * np.maximum(0.0, 1.0 - margins).mean()
    grad_w = w - (C / X.shape[0]) * (X[active].T @ t[active])
    grad_b = -(C / X.shape[0]) * float(t[active].sum())
    return float(objective), grad_w, float(grad_b), active

X = np.array([[-2., -1.], [-1., -1.], [1., 1.], [2., 1.]])
t = np.array([-1., -1., 1., 1.])
w0 = np.zeros(2)
J0, gw, gb, active = hinge_objective_and_subgradient(X, t, w0, 0.0, C=1.0)
w1, b1 = w0 - 0.2 * gw, -0.2 * gb
J1, _, _, _ = hinge_objective_and_subgradient(X, t, w1, b1, C=1.0)
assert active.shape == (4,) and gw.shape == (2,)
assert J1 < J0
print("objective", J0, "->", J1, "| active", active, "| w", w1)


## 5. Worked example: audit a boundary and its violations

Take $w=(1,0)$, $b=0$, rows $x=(-2,0),(-0.5,1),(0.75,-1),(2,0)$, and labels
$(-1,-1,+1,+1)$. Scores are $(-2,-0.5,0.75,2)$ and signed margins are
$(2,0.5,0.75,2)$. The middle two rows are correctly classified yet inside the margin, with
hinge losses $0.5$ and $0.25$. Correct classification alone does not imply zero hinge loss.

At canonical scaling, no row here has margin exactly 1, so calling the middle rows support
vectors from this primal snapshot would be premature; support-vector identity refers to the
fitted optimum/dual coefficients.

**Checkpoint 5A.** What is the mean hinge loss for the four rows?

**Checkpoint 5B.** Which rows are misclassified?


In [ ]:
margins = np.array([2.0, 0.5, 0.75, 2.0])
hinges = np.maximum(0.0, 1.0 - margins)
assert np.isclose(hinges.mean(), 0.1875, atol=ATOL, rtol=RTOL)
print("hinges", hinges, "mean", hinges.mean())


## 6. First explicit model-comparison axes

| Axis | Logistic regression | Linear SVM |
|---|---|---|
| supervision | binary labels | labels encoded $\{-1,+1\}$ |
| training objective | mean BCE plus optional regularization | norm penalty plus hinge violations |
| output | modeled probability via sigmoid | signed decision score and class |
| geometry | linear log-odds boundary | linear maximum-margin boundary |
| scaling | usually important | important because norms/distances set margins |
| probability/calibration | native probability, still check calibration | no native probability in basic formulation |
| interpretability | coefficient changes log-odds | coefficient defines margin normal |
| validation | classification metric plus probability metric | classification/margin metric; tune `C` in CV |

Neither wins universally. Choose logistic regression when probability estimates and calibration
matter; choose a linear SVM when margin-focused discrimination is primary. Both need leakage-safe
scaling and validation.

**Checkpoint 6A.** Which family has native modeled probabilities?

**Checkpoint 6B.** Name two axes on which the families agree.


## 7. Pitfalls, exam connections, and forward link

**Pitfalls.** Using labels `0/1` in $t_if_i$ breaks the negative class; using raw scores for
hinge activity loses the label sign; regularizing $b$ changes the stated objective; treating a
correct point inside the margin as inactive is wrong; and claiming a point is a support vector
without inspecting the fitted optimum confuses a snapshot with a solution.

**Exam connection.** A compact item may ask for signed margins, the active set, one subgradient
step, and an explanation of how `C` changes the trade-off. Pin the loss reduction and kink rule.

**Going deeper.** Session 3 reuses F7's duality and kernels. The comparison ledger gains nonlinear
capacity, kernel cost, and support-vector prediction structure.

**Checkpoint 7A.** Why does feature scaling affect an SVM even if it does not change labels?

**Checkpoint 7B.** State the hinge kink rule used in this unit.


## Checkpoint answers

**1A.** $2.5$, correct. **1B.** $+1$.

**2A.** $2/4=0.5$. **2B.** A support vector has a nonzero dual coefficient. Such rows usually
attain the margin in the nondegenerate hard-margin picture, but equality alone is insufficient
because a degenerate optimum can assign an equality row coefficient zero.

**3A.** $2,0.6,0,0$. **3B.** Toward fewer/more strongly penalized violations, with weaker
effective regularization.

**4A.** A negative-label row with a very negative correct score would be falsely marked active.
**4B.** `(7,)` and `(3,)`.

**5A.** $3/16=0.1875$. **5B.** None.

**6A.** Logistic regression. **6B.** Both are supervised linear boundaries and both commonly
need scaling/validation.

**7A.** Rescaling coordinates changes Euclidean norms and therefore the margin trade-off.
**7B.** Zero hinge contribution when signed margin equals 1.
